In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# 1. Import des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings

In [ ]:
import pandas as pd

# Essai avec différents encodages
try:
    df = pd.read_csv('all-data.csv', encoding='latin-1')
except:
    try:
        df = pd.read_csv('all-data.csv', encoding='iso-8859-1')
    except:
        df = pd.read_csv('all-data.csv', encoding='cp1252')

print(df.head())
print(f"\nNombre de phrases : {len(df)}")

Il n'y a pas de nom de colonne, donc on les rajoute

In [ ]:
df.columns = ['label', 'text']
df

In [ ]:
print(df['label'].unique())

In [ ]:
# On prend un échantillon de 200 phrases pour aller vite
sample = df.head(1000).copy()

# Vérification
print(sample.shape)
sample.head()

In [ ]:
print(sample['label'].unique())
print(sample['label'].value_counts())

In [ ]:
#On normalise les labels

def normalize_label(label):
    label = label.lower().strip()
    if label in ['positive', 'pos', 'label_1', '5 stars', '4 stars']:
        return 'positive'
    elif label in ['negative', 'neg', 'label_0', '1 star', '2 stars']:
        return 'negative'
    else:
        return 'neutral'

# BERT

In [ ]:
# Modèle généraliste
bert_pipeline = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

In [ ]:
bert_predictions = []

for text in sample['text']:
    result = bert_pipeline(text)[0]
    bert_predictions.append(normalize_label(result['label']))

sample['bert_prediction'] = bert_predictions


In [ ]:
print("\n" + "="*50)
print(" RESULTATS BERT GENERALISTE")
print("="*50)

bert_accuracy = accuracy_score(sample['label'], sample['bert_prediction'])
print(f"\n Accuracy BERT : {round(bert_accuracy*100, 2)}%")
print("\nRapport de classification :")
print(classification_report(
    sample['label'],
    sample['bert_prediction'],
    target_names=['negative', 'neutral', 'positive']
))

In [ ]:
# Histogramme + Matrice de confusion BERT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
sample['bert_prediction'].value_counts().plot(
    kind='bar', ax=axes[0],
    color=['red', 'gray', 'green'],
    edgecolor='black'
)
axes[0].set_title('BERT — Distribution des prédictions')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Nombre de phrases')
axes[0].tick_params(axis='x', rotation=0)

# Matrice de confusion
cm_bert = confusion_matrix(sample['label'], sample['bert_prediction'],
                           labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Reds',
            xticklabels=['neg', 'neu', 'pos'],
            yticklabels=['neg', 'neu', 'pos'], ax=axes[1])
axes[1].set_title('BERT — Matrice de confusion')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')

plt.tight_layout()
plt.show()

# FinBERT


In [ ]:
finbert_pipeline = pipeline("text-classification", model="ProsusAI/finbert")

In [ ]:
finbert_predictions = []

for text in sample['text']:
    result = finbert_pipeline(text)[0]
    finbert_predictions.append(normalize_label(result['label']))

sample['finbert_prediction'] = finbert_predictions

In [ ]:
print("\n" + "="*50)
print(" RESULTATS FINBERT")
print("="*50)

finbert_accuracy = accuracy_score(sample['label'], sample['finbert_prediction'])
print(f"\n Accuracy FinBERT : {round(finbert_accuracy*100, 2)}%")
print("\n Rapport de classification :")
print(classification_report(
    sample['label'],
    sample['finbert_prediction'],
    target_names=['negative', 'neutral', 'positive']
))

In [ ]:
# Histogramme + Matrice de confusion FinBERT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
sample['finbert_prediction'].value_counts().plot(
    kind='bar', ax=axes[0],
    color=['red', 'gray', 'green'],
    edgecolor='black'
)
axes[0].set_title('FinBERT — Distribution des prédictions')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Nombre de phrases')
axes[0].tick_params(axis='x', rotation=0)

# Matrice de confusion
cm_finbert = confusion_matrix(sample['label'], sample['finbert_prediction'],
                              labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_finbert, annot=True, fmt='d', cmap='Blues',
            xticklabels=['neg', 'neu', 'pos'],
            yticklabels=['neg', 'neu', 'pos'], ax=axes[1])
axes[1].set_title('FinBERT — Matrice de confusion')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')

plt.tight_layout()
plt.show()

# Comparaison

In [ ]:
print("COMPARAISON FINALE")

# Accuracy
bert_acc = round(bert_accuracy*100, 2)
finbert_acc = round(finbert_accuracy*100, 2)
print(f"BERT      : {bert_acc}%")
print(f"FinBERT   : {finbert_acc}%")
print(f"Meilleur  : {'FinBERT' if finbert_acc > bert_acc else 'BERT'}")

# Graphique
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['BERT', 'FinBERT'], [bert_acc, finbert_acc],
              color=['#FF6B6B', '#4ECDC4'], edgecolor='black', width=0.5)
ax.set_title('Comparaison Accuracy')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
for bar, val in zip(bars, [bert_acc, finbert_acc]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1, f'{val}%',
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

Finalement, FinBert est le meilleur modèle.
